# LangChain: Q&A over Documents
An example might be a tool that would allow you to query a product catalog for items of interest.

In [34]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

llm_model = "gpt-4.1-nano"

In [41]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from IPython.display import Markdown, display

In [42]:
loader = CSVLoader(file_path="assets//OutdoorClothingCatalog_1000.csv")
docs = loader.load()

In [43]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [44]:
llm = ChatOpenAI(
    model=llm_model,  # modern lightweight model
    temperature=0
)

In [45]:
prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.

Context:
{context}

Question:
{question}
""")

In [46]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [49]:
query = """
Please list all your shirts with sun protection in a table in markdown and summarize each one.
"""

response = rag_chain.invoke(query)

display(Markdown(response.content))

| Name                                              | Summary                                                                                                              |
|---------------------------------------------------|----------------------------------------------------------------------------------------------------------------------|
| Men's Plaid Tropic Shirt, Short-Sleeve            | Light, comfortable fishing shirt with UPF 50+ protection, quick-drying, wrinkle-free, with venting and pockets.     |
| Men's Tropical Plaid Short-Sleeve Shirt           | Relaxed fit, UPF 50+ protection, wrinkle-resistant polyester, venting, and pockets for sun protection and comfort.  |
| Sun Shield Shirt by                                | High-performance, abrasion-resistant sun shirt with UPF 50+, moisture-wicking, and quick-drying fabric.            |
| Men's TropicVibe Shirt, Short-Sleeve               | Lightweight, relaxed fit with UPF 50+, venting, pockets, and wrinkle resistance for hot weather sun protection.   |
| Tropical Breeze Shirt                              | Long-sleeve, breathable, lightweight UPF 50+ shirt with moisture-wicking, wrinkle resistance, and venting.       |